# Vibgyor — AI finishing pass on a free GPU

Turns the Blender renders of the **CAD-derived 3D model** into photoreal 360° spots and stills.
Every image keeps the model's geometry: RealVisXL V5 repaints materials and lighting while two
ControlNets fed with the model's exact **depth** and **edges** hold walls, doors, windows and furniture in place.

**Kaggle (recommended: 30 GPU hours a week, 29 GB RAM)**
1. *Create → New Notebook → File → Import Notebook* and pick this file.
2. Right panel → *Settings*: **Accelerator = GPU T4 x2** (or P100) and **Internet = On** (needs a phone-verified account).
3. *Add Input → Upload → New Dataset*: upload `outputs/ai_handoff/vibgyor_ai_inputs.zip` from the repo.
4. *Run All*. When it finishes, download `vibgyor_ai_results.zip` from the *Output* panel.

**Colab**: *Runtime → Change runtime type → T4 GPU*, then *Run all* and pick the zip when the upload box appears.
Colab's 12 GB of RAM is tight for this model set; use Kaggle if the session crashes while loading.

Back in the repo: `python -m pipeline.import_ai_results path/to/vibgyor_ai_results.zip`

In [ ]:
import socket, subprocess, sys
import torch

# Kaggle silently turns GPU and internet off for accounts without a verified phone number
if not torch.cuda.is_available():
    raise SystemExit("No GPU attached. Kaggle: verify your phone (Settings > Phone verification), "
                     "then Settings > Accelerator = GPU T4. Colab: Runtime > Change runtime type > T4 GPU.")
try:
    socket.create_connection(("huggingface.co", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet. Kaggle: verify your phone, then Settings > Internet = On.")
print("GPU:", torch.cuda.get_device_name(0))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "diffusers==0.42.1",
                       "transformers==5.22.0", "accelerate==1.15.0", "safetensors>=0.8"])

In [ ]:
import glob, os, pathlib, zipfile

ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = pathlib.Path("/kaggle/working" if ON_KAGGLE else "/content")

def find_inputs():
    roots = ["/kaggle/input", str(WORK)]
    for root in roots:
        hits = glob.glob(f"{root}/**/vibgyor_ai_inputs/jobs.json", recursive=True)
        if hits:
            return pathlib.Path(hits[0]).parent
    for root in roots:  # a zip that was not auto-extracted
        for z in glob.glob(f"{root}/**/vibgyor_ai_inputs*.zip", recursive=True):
            zipfile.ZipFile(z).extractall(WORK)
            return WORK / "vibgyor_ai_inputs"
    if not ON_KAGGLE:
        from google.colab import files
        for name in files.upload():
            zipfile.ZipFile(name).extractall(WORK)
        return WORK / "vibgyor_ai_inputs"
    raise FileNotFoundError("Add vibgyor_ai_inputs.zip as an input dataset")

INPUTS = find_inputs()
OUT = WORK / "results"
print("inputs:", INPUTS, "| outputs:", OUT)
print(open(INPUTS / "jobs.json").read()[:600])

## 1. Preview one spot
Renders the Type A drawing room 360° first (about 10 minutes including model download) so you can check the look before the full run.

In [ ]:
!python "{INPUTS}/finish_ai.py" --inputs "{INPUTS}" --out "{OUT}" --only pano_type_a_1_drawing_room --skip-existing
if _exit_code:
    raise SystemExit("finish_ai.py failed, see the log above")

In [ ]:
import sys
from IPython.display import display
from PIL import Image
sys.path.insert(0, str(INPUTS))
import numpy as np, finish_ai

pano = np.asarray(Image.open(OUT / "panoramas/pano_type_a_1_drawing_room.png").convert("RGB"))
display(Image.fromarray(pano).resize((1536, 768)))
for yaw in (0, 90, 180, 270):  # what the viewer shows when you look around
    display(Image.fromarray(finish_ai.equi_to_persp(pano, yaw, 0, 90, 768)))

## 2. Everything (16 spots + 20 stills)
About 2–3 hours on a T4. If the session drops, run this cell again: finished images are kept (`--skip-existing`).

In [ ]:
!python "{INPUTS}/finish_ai.py" --inputs "{INPUTS}" --out "{OUT}" --skip-existing
if _exit_code:
    raise SystemExit("finish_ai.py failed, see the log above")

In [ ]:
import shutil
archive = shutil.make_archive(str(WORK / "vibgyor_ai_results"), "zip", OUT)
print(archive, round(os.path.getsize(archive) / 2**20), "MB")
if not ON_KAGGLE:
    from google.colab import files
    files.download(archive)